# Valuation-to-Option Pricing Lab

This notebook is the analytical engine. Students edit the three CSV files in `data/`, run all cells, interpret the results, and publish the generated Plotly dashboard.

> Analyst targets support subjective valuation scenarios. They are not risk-neutral probabilities. The CRR lattice independently calculates the no-arbitrage model value.

## 1. Load and validate student inputs

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
ROOT=Path.cwd(); DATA=ROOT/'data'; DOCS=ROOT/'docs'; DOCS.mkdir(exist_ok=True)
inputs=pd.read_csv(DATA/'company_inputs.csv')
if len(inputs)!=1: raise ValueError('company_inputs.csv must contain exactly one firm row.')
M=inputs.iloc[0].to_dict()
nums=['spot','strike','maturity_years','risk_free_rate','dividend_yield','volatility','market_bid','market_ask','dcf_value','comps_value']
for k in nums: M[k]=float(M[k])
M['lattice_steps']=int(M['lattice_steps']); M['market_mid']=(M['market_bid']+M['market_ask'])/2
M['option_type']=str(M['option_type']).lower(); M['exercise_style']=str(M['exercise_style']).lower(); M['volatility_type']=str(M['volatility_type']).lower()
if M['spot']<=0 or M['strike']<=0 or M['maturity_years']<=0: raise ValueError('Spot, strike, and maturity must be positive.')
if not 0<=M['risk_free_rate']<=1: raise ValueError('risk_free_rate must be decimal form between 0 and 1. Enter 4.5% as 0.045.')
if not 0<=M['dividend_yield']<=1: raise ValueError('dividend_yield must be decimal form between 0 and 1.')
if not 0<M['volatility']<=3: raise ValueError('volatility must be annualized decimal form. Enter 28% as 0.28.')
if M['market_bid']>M['market_ask']: raise ValueError('market_bid cannot exceed market_ask.')
if M['option_type'] not in ['call','put']: raise ValueError('option_type must be call or put.')
if M['exercise_style'] not in ['european','american']: raise ValueError('exercise_style must be european or american.')
if M['volatility_type'] not in ['implied','historical']: raise ValueError('volatility_type must be implied or historical.')
if M['lattice_steps']<25: raise ValueError('Use at least 25 lattice steps; 100 or more is recommended.')
targets=pd.read_csv(DATA/'analyst_targets.csv')
required={'analyst','target','publication_date','horizon_months','source_name'}
if not required.issubset(targets.columns): raise ValueError(f'analyst_targets.csv must contain {sorted(required)}')
if len(targets)<5: raise ValueError('At least five analyst targets are required.')
if not targets.target.gt(0).all(): raise ValueError('Analyst targets must be positive.')
if targets.horizon_months.nunique()!=1: raise ValueError('All analyst targets must use one common horizon.')
sources=pd.read_csv(DATA/'source_documentation.csv')
if sources[['input_name','value','observation_date','source','method_or_note']].isna().any().any(): raise ValueError('Source documentation contains blank required fields.')
print('Validated company inputs')
display(pd.DataFrame([M]).T.rename(columns={0:'Imported value'}))

## 2. Maturity alignment

In [ ]:
target_horizon_years=float(targets.horizon_months.iloc[0])/12
horizon_gap=abs(target_horizon_years-M['maturity_years'])
horizon_status='PASS' if horizon_gap<=0.25 else 'WARNING'
print(f'Analyst horizon: {target_horizon_years:.2f} years')
print(f'Option maturity: {M["maturity_years"]:.2f} years')
print(f'Maturity gap: {horizon_gap:.2f} years — {horizon_status}')
if horizon_status=='WARNING': print('Explain this mismatch in the Model Risk section of the decision memo.')

## 3. Equity valuation and analyst distribution

In [ ]:
stats={'count':int(len(targets)),'mean':float(targets.target.mean()),'median':float(targets.target.median()),'std':float(targets.target.std(ddof=1)),'min':float(targets.target.min()),'max':float(targets.target.max())}
valuation={'Current market':M['spot'],'Comparable companies':M['comps_value'],'Student DCF':M['dcf_value'],'Analyst median':stats['median']}
display(pd.Series(valuation).rename('Value per share'))
display(pd.Series(stats).rename('Analyst target statistic'))

## 4. Reproducible valuation states

In [ ]:
quantiles=targets.target.quantile([.10,.50,.80,.95]).to_numpy(float)
states=pd.DataFrame({'state':['Bear','Base','Bull','Tail'],'terminal_price':quantiles,'subjective_weight':[.20,.45,.25,.10]})
assert np.isclose(states.subjective_weight.sum(),1) and states.subjective_weight.ge(0).all()
states

## 5. Payoffs and belief-weighted value

In [ ]:
def payoff(stock,strike,kind):
    stock=np.asarray(stock,float)
    return np.maximum(stock-strike,0) if kind=='call' else np.maximum(strike-stock,0)
states['option_payoff']=payoff(states.terminal_price,M['strike'],M['option_type'])
states['weighted_payoff']=states.subjective_weight*states.option_payoff
belief_value=float(np.exp(-M['risk_free_rate']*M['maturity_years'])*states.weighted_payoff.sum())
states

## 6. CRR lattice

In [ ]:
def crr(spot,strike,T,r,sigma,N,q=0,kind='call',american=False):
    dt=T/N; u=np.exp(sigma*np.sqrt(dt)); d=1/u; growth=np.exp((r-q)*dt); p=(growth-d)/(u-d); disc=np.exp(-r*dt)
    if not(d<growth<u and 0<p<1): raise ValueError('No-arbitrage condition failed. Review rate, yield, volatility, maturity, and steps.')
    j=np.arange(N+1); stock=spot*u**j*d**(N-j); values=payoff(stock,strike,kind)
    for n in range(N-1,-1,-1):
        values=disc*(p*values[1:n+2]+(1-p)*values[:n+1])
        if american:
            j=np.arange(n+1); stock=spot*u**j*d**(n-j); values=np.maximum(values,payoff(stock,strike,kind))
    return {'price':float(values[0]),'u':float(u),'d':float(d),'p_star':float(p),'dt':float(dt)}
lattice=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],M['option_type'],M['exercise_style']=='american')
lattice

## 7. Theoretical validation

In [ ]:
pvs=M['spot']*np.exp(-M['dividend_yield']*M['maturity_years']); pvk=M['strike']*np.exp(-M['risk_free_rate']*M['maturity_years'])
lo,hi=(max(pvs-pvk,0),pvs) if M['option_type']=='call' else (max(pvk-pvs,0),pvk)
call=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'call')['price']
put=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'put')['price']
parity=float((call-put)-(pvs-pvk)); steps=[25,50,100,200,500]
conv=[crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],n,M['dividend_yield'],M['option_type'])['price'] for n in steps]
validation={'valid_risk_neutral_probability':bool(0<lattice['p_star']<1),'inside_theoretical_bounds':bool(lo<=lattice['price']<=hi),'put_call_parity_pass':bool(abs(parity)<.05),'convergence_pass':bool(abs(conv[-1]-conv[-2])<.05),'parity_residual':parity,'convergence_gap':float(conv[-1]-conv[-2]),'maturity_alignment_status':horizon_status,'maturity_gap_years':horizon_gap}
assert all(validation[k] for k in ['valid_risk_neutral_probability','inside_theoretical_bounds','put_call_parity_pass','convergence_pass'])
validation

## 8. Export dashboard data

In [ ]:
results={'model':M,'analyst_targets':[float(x) for x in targets.target],'analyst_stats':stats,'valuation':valuation,'states':states.to_dict('records'),'valuation_weighted_expected_payoff':belief_value,'lattice':lattice,'validation':validation,'convergence':[{'steps':n,'price':float(x)} for n,x in zip(steps,conv)],'comparison':{'Valuation-weighted expected payoff':belief_value,'Risk-neutral lattice value':lattice['price'],'Observed market midpoint':M['market_mid']}}
(DOCS/'model_results.json').write_text(json.dumps(results,indent=2))
print('Exported docs/model_results.json')
results['comparison']

## 9. Decision memo

### Equity valuation
Replace this text with an interpretation of the current price, DCF, comparable value, and analyst median.

### Analyst distribution
Discuss central tendency, dispersion, outliers, dependence, and limitations.

### Option pricing
Report the belief-weighted expected payoff, CRR value, bid, midpoint, and ask.

### Model risk
Discuss maturity alignment, volatility choice, dividends, liquidity, bid-ask spread, subjective weights, and model specification.

### Conclusion
State what the evidence supports without automatically characterizing the option as mispriced.